# 🚀 Zalo AI Challenge 2026: DataMate — Pilot Baseline (Qwen2.5-Coder-7B-Instruct)

> **Mục tiêu:** Đánh giá pilot baseline trên **30 câu train** phân tầng (6 shop x 5 câu), sử dụng model `Qwen/Qwen2.5-Coder-7B-Instruct` gốc (**zero-shot, chưa fine-tune**) trên GPU A100.
>
> **Kỷ luật dữ liệu & Quy chế:**
> 1. Dữ liệu thô `data/` là **STRICTLY READ-ONLY**.
> 2. File ground truth `eval_gold/pilot_gold_30.jsonl` được **cô lập hoàn toàn**, nằm ngoài tầm truy cập của sandbox.
> 3. Ngân sách context budget: `16.384` token, sinh tối đa `2.048` token/turn.
> 4. Báo cáo bằng số liệu thật: Đo điểm PA, thời gian thực thi, VRAM và phân loại lỗi thực tế.


## Bước 1: Kiểm tra phần cứng & CUDA
Xác nhận môi trường đang kết nối GPU A100 (khuyến nghị) hoặc V100/L4.


In [ ]:
!nvidia-smi
import torch
print('PyTorch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU device name:', torch.cuda.get_device_name(0))
    print(f'Total VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB')


## Bước 2: Kết nối Google Drive & Thiết lập Workspace
Mount Google Drive và đồng bộ mã nguồn sang ổ SSD cục bộ `/content/datamate` để đạt tốc độ I/O nhanh nhất.


In [ ]:
from google.colab import drive
import os, shutil

drive.mount('/content/drive')

gdrive_project = '/content/drive/MyDrive/Colab Notebooks/ZALO-AI-CHALLENGE-2026'
local_work = '/content/datamate'

assert os.path.exists(gdrive_project), f'Không tìm thấy {gdrive_project} trên Google Drive!'

# Sao chép sang SSD cục bộ /content/datamate (bỏ qua git/cache để tăng tốc)
!mkdir -p /content/datamate
!rsync -av --exclude='.git' --exclude='wandb' --exclude='__pycache__' "$gdrive_project/" /content/datamate/
%cd /content/datamate

print('✅ Workspace đã sẵn sàng tại /content/datamate!')


## Bước 3: Cài đặt Dependencies & Chạy Regression Test
Cài các thư viện cần thiết và chạy unit test để xác nhận logic thời gian và sandbox đều hoạt động chuẩn.


In [ ]:
!pip install -q transformers accelerate pandas python-calamine pyyaml tqdm pytest

# Chạy bộ test kiểm định cốt lõi
!python -m pytest tests/test_time_resolver.py tests/test_sandbox_protocol.py tests/test_cli_and_integration.py


## Bước 4: Tải Weights Qwen2.5-Coder-7B-Instruct về máy
Tải toàn bộ weights về thư mục `/content/models/Qwen2.5-Coder-7B-Instruct` qua `snapshot_download` (Python native) để model chạy ở chế độ offline hoàn toàn.


In [ ]:
import os
from huggingface_hub import snapshot_download

model_dir = '/content/models/Qwen2.5-Coder-7B-Instruct'
os.makedirs(model_dir, exist_ok=True)

print(f'Bắt đầu tải weights Qwen/Qwen2.5-Coder-7B-Instruct vào {model_dir}...')
snapshot_download(
    repo_id='Qwen/Qwen2.5-Coder-7B-Instruct',
    local_dir=model_dir,
    local_dir_use_symlinks=False,
    resume_download=True,
)

# Xác minh file weights
files = os.listdir(model_dir)
print(f'Tổng số files đã tải: {len(files)}')
assert 'config.json' in files, 'Thiếu config.json!'
print('✅ Model weights đã tải và xác minh thành công!')


## Bước 5: Cấu hình Suy luận cho A100
Cập nhật `configs/baseline_7b_a100.yaml` để trỏ đúng đường dẫn model weights và thư mục sandbox `/tmp/work_pilot`.


In [ ]:
import yaml
cfg_path = 'configs/baseline_7b_a100.yaml'
with open(cfg_path, 'r', encoding='utf-8') as f:
    cfg = yaml.safe_load(f)

cfg['model']['local_model_dir'] = '/content/models/Qwen2.5-Coder-7B-Instruct'
cfg['paths']['work_dir'] = '/tmp/work_pilot'

with open(cfg_path, 'w', encoding='utf-8') as f:
    yaml.dump(cfg, f, allow_unicode=True, indent=2)

print('Config baseline A100:')
print(yaml.dump(cfg, allow_unicode=True, indent=2))


## Bước 6: Smoke Test (2 câu đầu)
Chạy thử nghiệm 2 câu đầu tiên trong tập 30 câu để kiểm tra:
- Model nạp vào VRAM ổn định.
- Sinh code Python, sandbox thực thi trên dữ liệu thật và trả JSON protocol.
- Ghi log câu hỏi đúng chuẩn.


In [ ]:
!python scripts/run_inference.py \
    --config configs/baseline_7b_a100.yaml \
    --questions-file result/baseline_7b_pilot/pilot_questions_30.jsonl \
    --data-dir data/train \
    --output result/baseline_7b_pilot/smoke_submission.jsonl \
    --logs-dir result/baseline_7b_pilot/smoke_logs \
    --limit 2

print('\n--- Kết quả Smoke Test ---')
with open('result/baseline_7b_pilot/smoke_submission.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        print(line.strip())


## Bước 7: Chạy chính thức toàn bộ 30 câu (Frozen Baseline Run)
> ⚠️ **Kỷ luật:** Cấu hình và prompt được đóng băng. Chạy đủ 30 câu liên tục. Giữ nguyên tất cả kết quả (kể cả timeout/OOM nếu có) vào báo cáo.


In [ ]:
import time
t0 = time.time()

!python scripts/run_inference.py \
    --config configs/baseline_7b_a100.yaml \
    --questions-file result/baseline_7b_pilot/pilot_questions_30.jsonl \
    --data-dir data/train \
    --output result/baseline_7b_pilot/submission.jsonl \
    --logs-dir result/baseline_7b_pilot/logs \
    --log-file result/baseline_7b_pilot/run.log

elapsed = time.time() - t0
print(f'\n✅ Hoàn thành 30 câu trong {elapsed:.1f}s (trung bình {elapsed/30:.1f}s/câu).')


## Bước 8: Chấm điểm Penalized Accuracy (PA)
Chấm dự đoán `submission.jsonl` với đúng 30 nhãn trong `eval_gold/pilot_gold_30.jsonl`.
- Đúng: `+1.0`
- Abstain: `0.0`
- Sai: `-1.0`


In [ ]:
!python scripts/evaluate_local.py \
    --pred result/baseline_7b_pilot/submission.jsonl \
    --gold eval_gold/pilot_gold_30.jsonl


## Bước 9: Đồng bộ kết quả về Google Drive
Lưu toàn bộ `submission.jsonl`, `logs/`, `run.log` về Google Drive để đồng bộ tức thì với máy local.


In [ ]:
!mkdir -p "$gdrive_project/result/baseline_7b_pilot"
!cp -r result/baseline_7b_pilot/* "$gdrive_project/result/baseline_7b_pilot/"
print('✅ Toàn bộ artifacts và logs đã được lưu vào Google Drive!')
print(f'Đường dẫn Drive: {gdrive_project}/result/baseline_7b_pilot/')
